# 01 — Exploratory analysis

Runs only on canonical tables produced by an approved adapter. While the dataset gate is blocked it reports that and stops. Nothing is simulated.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from aerolife.config import load_config, gate_status, resolve
cfg = load_config(ROOT / 'configs' / 'config.yaml')
canon = resolve(cfg, 'canonical_dir')
READY = gate_status(cfg) == 'approved' and (canon / 'observations.csv').exists()
print('gate:', gate_status(cfg), '| canonical data present:', (canon / 'observations.csv').exists())
if not READY:
    print('BLOCKED: no approved dataset; see FEASIBILITY.md. Remaining cells are skipped.')

gate: blocked | canonical data present: False
BLOCKED: no approved dataset; see FEASIBILITY.md. Remaining cells are skipped.


## Units, lifetimes, censoring, operating conditions

In [2]:
if READY:
    import pandas as pd
    from aerolife.pipeline import load_canonical
    obs, units = load_canonical(canon)
    display(units)
    print(obs.groupby('unit_id').time_value.agg(['count', 'max']).describe())

## Channel audit: missing values, constant channels, ranges (all units; descriptive only, not used for fitting)

In [3]:
if READY:
    from aerolife.config import require_dataset
    from aerolife.preprocessing import audit
    spec = require_dataset(cfg)
    display(audit(obs, spec['input_columns']))

## Recording gaps and schema check

In [4]:
if READY:
    from aerolife.schema import validate_observations, validate_units
    r = validate_observations(obs, spec['input_columns']); print(r.errors, r.warnings[:20])
    print(validate_units(units, obs).errors)

## Channel trends for training units only (avoid peeking at test units)

In [5]:
if READY and (resolve(cfg, 'splits_dir') / 'split_manifest.json').exists():
    import matplotlib.pyplot as plt
    from aerolife.splits import load_manifest
    train = load_manifest(resolve(cfg, 'splits_dir') / 'split_manifest.json')['split']['train']
    for c in spec['input_columns'][:6]:
        fig, ax = plt.subplots(figsize=(7, 2.5))
        for uid, g in obs[obs.unit_id.isin(train)].groupby('unit_id'):
            ax.plot(g.time_value, g[c], lw=0.8)
        ax.set(title=c, xlabel=f"elapsed time ({spec['time_unit']})"); plt.show()